In [1]:
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import os
import json


In [2]:
search = TavilySearchResults()

@tool
def search_tool(query: str):
    """
        Search the web for information using Tavily API.
    
        :param query: The search query string
        :return: Search results related to the query
        """
    return search.invoke(query)

C:\Users\ntmtinh\AppData\Local\Temp\ipykernel_6008\3497864574.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search = TavilySearchResults()


In [3]:
search_tool.invoke("What's the weather like in Tokyo today?")

[{'title': 'Tokyo Weather Conditions: Temperature | 30 Days Forecast',
  'url': 'https://www.aqi.in/weather/us/japan/tokyo/tokyo',
  'content': "2. What are the current weather parameters in Tokyo today?\n\nHere is a complete overview of Tokyo's current weather conditions as of 01:30 PM 23 June 2026:\nCondition : Partly Cloudy\nTemperature : 24°C Pleasant (Feels like 26°C)\nHumidity : 61%\nWind Speed : 15.5 km/h from ENE (69 degrees), Gust 5.2 m/s\nUV Index : 3.6 (Moderate\nAir Pressure : 1010 mb\nCloud Cover : 50%\nVisibility : 10 km\nPrecipitation : 0 mm (22% chance of rain)\n\n3. What is the 10-day weather forecast for Tokyo from 01:30 PM 23 June 2026? [...] From 01:30 PM 23 June 2026, Tokyo's 10-day forecast shows this trend:\nToday (Jun. 23) : Temp. 21°C, Hum. 67% and Patchy rain nearby condition.\nWednesday (Jun. 24) : Temp. 22°C, Hum. 68% and Patchy rain nearby condition.\nThursday (Jun. 25) : Temp. 21°C, Hum. 82% and Rain condition.\nFriday (Jun. 26) : Temp. 24°C, Hum. 81% and 

In [4]:
@tool
def recommend_clothing(weather: str) -> str:
    """
    Returns a clothing recommendation based on the provided weather description.

    This function examines the input string for specific keywords or temperature indicators 
    (e.g., "snow", "freezing", "rain", "85°F") to suggest appropriate attire. It handles 
    common weather conditions like snow, rain, heat, and cold by providing simple and practical 
    clothing advice.

    :param weather: A brief description of the weather (e.g., "Overcast, 64.9°F")
    :return: A string with clothing recommendations suitable for the weather
    """
    weather = weather.lower()
    if "snow" in weather or "freezing" in weather:
        return "Wear a heavy coat, gloves, and boots."
    elif "rain" in weather or "wet" in weather:
        return "Bring a raincoat and waterproof shoes."
    elif "hot" in weather or "85" in weather:
        return "T-shirt, shorts, and sunscreen recommended."
    elif "cold" in weather or "50" in weather:
        return "Wear a warm jacket or sweater."
    else:
        return "A light jacket should be fine."

In [5]:
tools=[search_tool,recommend_clothing]

tools_by_name={ tool.name:tool for tool in tools}

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool

In [7]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


SystemPrompt

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, AIMessage, ToolMessage

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful AI assistant that thinks step-by-step and uses tools when needed.
    
    When responding to queries:
    1. First, think about what information you need
    2. Use available tools if you need current data or specific capabilities  
    3. Provide clear, helpful responses based on your reasoning and any tool results
    
    Always explain your thinking process to help users understand your approach.
    """),
    MessagesPlaceholder(variable_name="scratch_pad")
])

In [9]:
model_react = chat_prompt | model.bind_tools(tools)

In [10]:
from typing import (TypedDict, Annotated, Sequence)
from langgraph.graph.message import add_messages


class AgentState(TypedDict):
    """The state of the agent."""
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [11]:
state: AgentState = {"messages": []}

state["messages"] = add_messages(state["messages"], [HumanMessage(content="Hi")])
print("After greeting:", state["messages"])

state["messages"] = add_messages(state["messages"], [HumanMessage(content="Weather in NYC?")])
print("After question:", state)


After greeting: [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='82ead91b-3f11-4c06-9d09-ee76295dc9fe')]
After question: {'messages': [HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='82ead91b-3f11-4c06-9d09-ee76295dc9fe'), HumanMessage(content='Weather in NYC?', additional_kwargs={}, response_metadata={}, id='fef76ae0-b5fc-4841-8853-0ac095834604')]}


Manual React Execution

In [12]:
dummy_state: AgentState = {
    "messages": [HumanMessage("What's the weather like in Zurich, and what should I wear based on the temperature?")]
}

response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state["messages"] = add_messages(dummy_state["messages"], [response])


In [13]:
tool_call = response.tool_calls[-1]
print("Tool call:", tool_call)

tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
print("Tool result preview:", tool_result[0]['title'])

tool_message = ToolMessage(
    content = json.dumps(tool_result),
    name = tool_call["name"],
    tool_call_id = tool_call["id"]
)
dummy_state["messages"] = add_messages(dummy_state["messages"], [tool_message])

Tool call: {'name': 'search_tool', 'args': {'query': 'weather in Zurich'}, 'id': '9578a894-e1d7-4f54-9820-5da419ae8933', 'type': 'tool_call'}
Tool result preview: Zurich weather in June 2026 | Zurich 14 day weather


Process result and next action

In [14]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
dummy_state['messages'] = add_messages(dummy_state['messages'], [response])

# check if the model wants to use another tool
if response.tool_calls:
    tool_call = response.tool_calls[0]
    tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
    tool_message = ToolMessage(
        content=json.dumps(tool_result),
        name=tool_call["name"],
        tool_call_id=tool_call["id"]
    )
    dummy_state['messages'] = add_messages(dummy_state['messages'], [tool_message])

In [15]:
response = model_react.invoke({"scratch_pad": dummy_state["messages"]})
print("Final response generated:", response.content is not None)
print("More tools needed:", bool(response.tool_calls))

Final response generated: True
More tools needed: False


Tool execution node

In [16]:
def tool_node(state: AgentState):
    """Execute all tool calls from the last message in the state."""
    outputs = []
    for tool_call in state['messages'][-1].tool_calls:
        tool_result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        outputs.append(ToolMessage(
            content=json.dumps(tool_result),
            name=tool_call["name"],
            tool_call_id=tool_call["id"]
        ))

    return {"messages": outputs}

In [17]:
def call_model(state: AgentState):
    """Invoke the model with the current conversation state."""
    response = model_react.invoke({"scratch_pad": state["messages"]})
    return {"messages": [response]}

In [18]:
def should_continue(state: AgentState):
    messages = state['messages']
    last_message = messages[-1]
    if not last_message.tool_calls:
        return "end"
    else:
        return "continue"

In [19]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

# Define the two nodes we will cycle between
workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.add_edge("tools", "agent")

workflow.add_conditional_edges(
    "agent",
    should_continue,
    {
        "continue": "tools",  # If tools needed, go to tools node
        "end": END,          # If done, end the conversation
    },
)

# Set entry point
workflow.set_entry_point("agent")

# Compile the graph
graph = workflow.compile()

In [23]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

In [24]:
def print_stream(stream):
    """Helper function for formatting the stream nicely."""
    for s in stream:
        message = s["messages"][-1]
        if isinstance(message, tuple):
            print(message)
        else:
            message.pretty_print()

inputs = {"messages": [HumanMessage(content="What's the weather like in Zurich, and what should I wear based on the temperature?")]}

print_stream(graph.stream(inputs, stream_mode="values"))

================================ Human Message =================================

What's the weather like in Zurich, and what should I wear based on the temperature?
================================== Ai Message ==================================
Tool Calls:
  search_tool (77bd5f5e-905f-4baa-a4e9-42b3b501fb17)
 Call ID: 77bd5f5e-905f-4baa-a4e9-42b3b501fb17
  Args:
    query: weather in Zurich
================================= Tool Message =================================
Name: search_tool

[{"title": "Zurich weather in June 2026 | Zurich 14 day weather", "url": "https://www.weather25.com/europe/switzerland/zurich?page=month&month=June", "content": "weather25.com \n\nUnited States England Australia Canada\n\n\u00b0F \u00b0C\n\nWeather in June 2026\n\nRemove from your favorite locations Add to my locations\n\nShare\n\n1. Home\n2. Europe\n3. weather in SwitzerlandSwitzerland\n4. Zurich\n5. June\n\nLocation was added to My Locations\n\nLocation was removed from My Locations\n\n# Zurich we